# Semana 5 — Terminologías clínicas: SNOMED CT, LOINC y el servidor terminológico nacional (NCTS)

**Objetivo:** auditar los códigos clínicos de tus datos (A) sin conexión, con los dígitos verificadores oficiales de SNOMED CT y LOINC, y (B) contra el **National Terminology Server** de Australia con operaciones FHIR (`$lookup`, `$validate-code`, `$expand` con ECL), y (C) construir **conjuntos de conceptos** reutilizables para la semana 7.

**Productos reales usados:** NCTS National Terminology Server (Ontoserver, Australian Digital Health Agency), SNOMED CT-AU, LOINC.

**Al terminar tendrás:** `data/reference/terminology_qa.parquet` y `data/reference/concept_sets.parquet`.

**Requisitos:** semana 3 y tus credenciales de cliente NCTS (Client ID y Client Secret). Sin credenciales, las Partes A y C funcionan igual; la Parte B se omite.

### Celda de configuración del entorno (se repite al inicio de cada semana)

**Explicación línea por línea**

- `import os`, `import sys`, `from pathlib import Path`: cargan tres módulos de la librería estándar. `os` lee variables de entorno, `sys` informa la versión de Python y `Path` representa rutas de archivos como objetos. **Por qué así:** `Path` evita concatenar rutas con texto (`"a" + "/" + "b"`), que falla entre sistemas operativos. **En la práctica real** todo código de datos moderno usa `pathlib`.
- `try: import google.colab` … `except ImportError`: intenta importar un módulo que solo existe en Colab. Si la importación funciona, `IN_COLAB = True`; si lanza `ImportError`, estamos en otra máquina. **Por qué importa:** el mismo notebook corre en Colab y en tu laptop sin editarlo. **Cuándo se usa:** siempre que un código deba ser portable entre entornos.
- `from google.colab import drive` y `drive.mount("/content/drive")`: conectan tu Google Drive al sistema de archivos de Colab. **Por qué:** el disco de Colab se borra al cerrar la sesión; en Drive los datos de la semana 1 siguen ahí el lunes siguiente. La primera vez Colab te pedirá autorizar el acceso.
- `PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")`: carpeta raíz del proyecto dentro de tu Drive.
- `Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()`: fuera de Colab usa la variable de entorno `HEALTHETL_PROJECT_DIR` si existe; si no, la carpeta actual (`"."`). `.resolve()` la convierte en ruta absoluta. **En la práctica real** la configuración se pasa por variables de entorno, nunca escrita a mano dentro del código (principio *12-factor app*).
- `DATA_DIR = PROJECT_DIR / "data"`: el operador `/` de `pathlib` une rutas de forma segura.
- `DATA_DIR.mkdir(parents=True, exist_ok=True)`: crea la carpeta y sus padres; `exist_ok=True` evita un error si ya existe. Esto hace la celda **idempotente**: puedes ejecutarla 1 o 10 veces con el mismo resultado, una propiedad central de todo pipeline ETL.
- Los tres `print`: confirman versión de Python, entorno y ruta. Registrar el entorno al inicio es un hábito de reproducibilidad.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # este módulo solo existe dentro de Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")
else:
    PROJECT_DIR = Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("¿Colab?:", IN_COLAB)
print("Carpeta de datos:", DATA_DIR)

### Paso 1 — Inventario de códigos usados en tus datos

**Explicación línea por línea**

- `import pandas as pd`, `REF_DIR = DATA_DIR / "reference"`: carpeta para tablas de referencia (terminología, conjuntos de conceptos).
- `cond = pd.read_parquet(..., columns=["snomed_code", "description", "patient_id"])`: condiciones de silver.
- `.groupby(["snomed_code", "description"]).agg(n_patients=("patient_id", "nunique"))`: una fila por código con el número de pacientes distintos. `agg(nombre=(columna, función))` es la sintaxis de "agregación con nombre".
- Lo mismo para observaciones con `loinc_code`.
- **Por qué empezar por el inventario:** en la práctica real, la auditoría terminológica se prioriza por impacto: un código mal usado en 500 pacientes importa más que uno en 2.

In [ ]:
import pandas as pd

REF_DIR = DATA_DIR / "reference"
REF_DIR.mkdir(parents=True, exist_ok=True)

cond = pd.read_parquet(DATA_DIR / "silver" / "conditions.parquet", columns=["snomed_code", "description", "patient_id"])
snomed_inv = (cond.groupby(["snomed_code", "description"])
                  .agg(n_patients=("patient_id", "nunique"))
                  .reset_index().sort_values("n_patients", ascending=False))

obs = pd.read_parquet(DATA_DIR / "silver" / "observations.parquet", columns=["loinc_code", "description", "patient_id"])
loinc_inv = (obs.groupby(["loinc_code", "description"])
                .agg(n_patients=("patient_id", "nunique"))
                .reset_index().sort_values("n_patients", ascending=False))
print("Códigos de condición:", snomed_inv["snomed_code"].nunique(), "| códigos de observación:", loinc_inv["loinc_code"].nunique())
snomed_inv.head()

## Parte A — Validación sin conexión con dígitos verificadores

### Paso 2 — Verhoeff para identificadores SNOMED CT (SCTID)

El último dígito de todo SCTID es un dígito verificador calculado con el algoritmo de Verhoeff (comprobación diédrica D5). Detecta la mayoría de errores de tipeo y de copiado (por ejemplo, Excel convirtiendo un código largo en notación científica).

**Explicación línea por línea**

- `D = [...]`: tabla de multiplicación del grupo diédrico D5 (10 × 10), tal como la publica la especificación de SNOMED CT.
- `F = [list(range(10)), [1, 5, 7, 6, 2, 8, 3, 0, 9, 4]]`: las dos primeras filas de la función de permutación F.
- `for i in range(2, 8): F.append([F[i - 1][F[1][j]] for j in range(10)])`: las filas 2 a 7 se **derivan** aplicando la permutación de la fila 1 sobre la fila anterior. **Por qué derivar en vez de copiar 8 filas:** copiar tablas a mano introduce errores difíciles de ver; derivarlas de la definición es más seguro. (Al preparar este notebook, una fila copiada de memoria estaba mal y hacía fallar códigos válidos; la derivación lo corrigió).
- `def verhoeff_ok(code):` recorre los dígitos de derecha a izquierda (`reversed`); `i` es la posición contando desde 0 en el dígito verificador.
- `c = D[c][F[i % 8][int(ch)]]`: en cada paso combina el acumulado con el dígito permutado según su posición (`% 8` porque F se repite cada 8 posiciones).
- `return c == 0`: el identificador es válido si el acumulado termina en 0.
- `code.isdigit() and 6 <= len(code) <= 18`: un SCTID tiene entre 6 y 18 dígitos.
- `def partition(code): return code[-3:-2] + code[-2:-1]`: los dos dígitos antes del verificador son el **identificador de partición**: `00` = concepto en formato corto (Edición Internacional), `10` = concepto en formato largo (con espacio de nombres de una extensión nacional, como la de EE. UU. o la australiana).
- Las pruebas con `assert` usan ejemplos de la especificación oficial: `100005` es el SCTID de concepto más bajo posible.

In [ ]:
D = [
    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [1, 2, 3, 4, 0, 6, 7, 8, 9, 5],
    [2, 3, 4, 0, 1, 7, 8, 9, 5, 6], [3, 4, 0, 1, 2, 8, 9, 5, 6, 7],
    [4, 0, 1, 2, 3, 9, 5, 6, 7, 8], [5, 9, 8, 7, 6, 0, 4, 3, 2, 1],
    [6, 5, 9, 8, 7, 1, 0, 4, 3, 2], [7, 6, 5, 9, 8, 2, 1, 0, 4, 3],
    [8, 7, 6, 5, 9, 3, 2, 1, 0, 4], [9, 8, 7, 6, 5, 4, 3, 2, 1, 0],
]
F = [list(range(10)), [1, 5, 7, 6, 2, 8, 3, 0, 9, 4]]
for i in range(2, 8):
    F.append([F[i - 1][F[1][j]] for j in range(10)])


def verhoeff_ok(code):
    if not (code.isdigit() and 6 <= len(code) <= 18):
        return False
    c = 0
    for i, ch in enumerate(reversed(code)):
        c = D[c][F[i % 8][int(ch)]]
    return c == 0


def partition(code):
    return code[-3:-1] if len(code) >= 3 else None


assert verhoeff_ok("100005") and verhoeff_ok("1290023401004") and verhoeff_ok("11000001102")
assert not verhoeff_ok("100006")
print("Verhoeff OK con los ejemplos oficiales")

### Paso 3 — Mod 10 para códigos LOINC

Un código LOINC es un número seguido de guion y un dígito verificador Mod 10 (por ejemplo `8480-6`). El manual de LOINC describe el algoritmo en 8 pasos; con el ejemplo oficial, el número `12345` tiene dígito verificador `5`.

**Explicación línea por línea**

- `def loinc_check_digit(body):` calcula el dígito para la parte numérica.
- `for i, ch in enumerate(reversed(body)):` recorre de derecha a izquierda.
- `if i % 2 == 0: d *= 2; d = d // 10 + d % 10`: las posiciones impares contando desde la derecha (1ª, 3ª, ...) se duplican y, si el resultado tiene dos cifras, se suman sus cifras (`14` → `1 + 4 = 5`). Es equivalente a los pasos 2 a 6 del manual.
- `return (10 - total % 10) % 10`: distancia hasta el siguiente múltiplo de 10; el `% 10` final convierte un 10 en 0.
- `def loinc_ok(code):` exige el formato `dígitos-dígito` con `re.fullmatch(r"\d+-\d", code)` y compara.
- `assert loinc_check_digit("12345") == 5`: el ejemplo del manual. `8480-6` es la presión sistólica.

In [ ]:
import re


def loinc_check_digit(body):
    total = 0
    for i, ch in enumerate(reversed(body)):
        d = int(ch)
        if i % 2 == 0:
            d *= 2
            d = d // 10 + d % 10
        total += d
    return (10 - total % 10) % 10


def loinc_ok(code):
    if not re.fullmatch(r"\d+-\d", code):
        return False
    body, check = code.split("-")
    return loinc_check_digit(body) == int(check)


assert loinc_check_digit("12345") == 5 and loinc_ok("8480-6") and not loinc_ok("8480-5")
print("Mod 10 OK con el ejemplo del manual de LOINC")

### Paso 4 — Aplicar ambas validaciones al inventario

**Explicación línea por línea**

- `snomed_inv["verhoeff_ok"] = snomed_inv["snomed_code"].map(verhoeff_ok)`: `map` aplica la función a cada código.
- `snomed_inv["partition"] = ...map(partition)`: añade la partición para ver qué códigos vienen de extensiones (`10`).
- `loinc_inv["loinc_format_ok"] = ...map(loinc_ok)`: valida formato y dígito LOINC.
- `loinc_inv["looks_snomed"] = loinc_inv["loinc_code"].map(verhoeff_ok)`: detecta códigos que en realidad parecen SCTID dentro de una columna que debería ser LOINC: un **error de sistema de codificación** frecuente cuando se mezclan fuentes.
- Los `print` listan los códigos que fallan. Verás códigos propios del generador (`QALY`, `DALY`, `QOLS`, `99999-x`) que no son LOINC, y al menos un SCTID con dígito verificador inválido. **En la práctica real** estos hallazgos se reportan al proveedor del dato y, mientras tanto, se excluyen del análisis con una regla documentada.

In [ ]:
snomed_inv["verhoeff_ok"] = snomed_inv["snomed_code"].map(verhoeff_ok)
snomed_inv["partition"] = snomed_inv["snomed_code"].map(partition)
loinc_inv["loinc_format_ok"] = loinc_inv["loinc_code"].map(loinc_ok)
loinc_inv["looks_snomed"] = loinc_inv["loinc_code"].map(verhoeff_ok)

print("SNOMED: códigos con dígito verificador inválido")
print(snomed_inv.loc[~snomed_inv["verhoeff_ok"], ["snomed_code", "description", "n_patients"]].to_string(index=False))
print("\nParticiones:", snomed_inv["partition"].value_counts().to_dict())
print("\nObservaciones cuyo código NO es LOINC válido:")
print(loinc_inv.loc[~loinc_inv["loinc_format_ok"], ["loinc_code", "description", "n_patients", "looks_snomed"]].to_string(index=False))

## Parte B — National Terminology Server (NCTS) por FHIR

### Paso 5 — Credenciales sin escribirlas en el código

**Explicación línea por línea**

- `NCTS_TOKEN_URL` y `NCTS_FHIR_BASE`: el endpoint de tokens OAuth2 y la base FHIR R4 del National Terminology Server, según la guía para implementadores del NCTS. Se leen de variables de entorno con valores por defecto, para poder apuntar a otro servidor (por ejemplo, un Ontoserver propio) sin editar código.
- `def get_secret(name):` en Colab usa **Secrets** (icono de llave en la barra lateral): guarda allí `NCTS_CLIENT_ID` y `NCTS_CLIENT_SECRET` y concede acceso a este notebook. Fuera de Colab, usa variables de entorno.
- `from google.colab import userdata` y `userdata.get(name)`: API oficial de Colab para leer un secreto. Los secretos **no** se guardan en el `.ipynb` ni se comparten al compartir el notebook.
- `except Exception as exc:` si el secreto no existe o el notebook no tiene permiso, devuelve `None` y explica por qué, en vez de detener el notebook.
- `ONLINE = bool(CLIENT_ID and CLIENT_SECRET)`: la Parte B solo corre con credenciales.
- **Regla de práctica real:** nunca escribas credenciales en un notebook. Un `.ipynb` subido a GitHub con un secreto se considera filtrado aunque lo borres después (queda en el historial de git).

In [ ]:
import time
import requests

NCTS_TOKEN_URL = os.environ.get("NCTS_TOKEN_URL", "https://api.healthterminologies.gov.au/oauth2/token")
NCTS_FHIR_BASE = os.environ.get("NCTS_FHIR_BASE", "https://api.healthterminologies.gov.au/integration/R4/fhir")


def get_secret(name):
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except Exception as exc:
            print(f"Secreto {name} no disponible ({type(exc).__name__})")
            return None
    return os.environ.get(name)


CLIENT_ID = get_secret("NCTS_CLIENT_ID")
CLIENT_SECRET = get_secret("NCTS_CLIENT_SECRET")
ONLINE = bool(CLIENT_ID and CLIENT_SECRET)
print("Modo en línea con NCTS:", ONLINE)

### Paso 6 — Token OAuth2 (*client credentials*) con caché

**Explicación línea por línea**

- `_token = {"value": None, "expires_at": 0.0}`: guarda el token y su vencimiento en memoria.
- `def get_token():` devuelve un token válido.
- `if _token["value"] and time.time() < _token["expires_at"] - 60:` reutiliza el token si le queda más de un minuto. **Por qué caché:** pedir un token nuevo en cada consulta multiplica las llamadas y puede activar límites del servidor.
- `requests.post(NCTS_TOKEN_URL, data={...}, timeout=30)`: `data=` envía un formulario `application/x-www-form-urlencoded` con `grant_type=client_credentials`, `client_id` y `client_secret`, como indica la guía del NCTS. Es el flujo OAuth2 para sistemas sin usuario humano (máquina a máquina).
- `payload["access_token"]` y `payload.get("expires_in", 300)`: campos estándar de una respuesta OAuth2; si el servidor no informa la duración se asumen 300 s.
- `session.headers["Accept"] = "application/fhir+json"`: igual que en la semana 4.
- `def tx_get(path, params):` hace un `GET` autenticado con `Authorization: Bearer <token>` y devuelve el JSON. Todas las operaciones terminológicas pasan por aquí.

In [ ]:
_token = {"value": None, "expires_at": 0.0}
session = requests.Session()
session.headers["Accept"] = "application/fhir+json"


def get_token():
    if _token["value"] and time.time() < _token["expires_at"] - 60:
        return _token["value"]
    r = requests.post(
        NCTS_TOKEN_URL,
        data={"grant_type": "client_credentials", "client_id": CLIENT_ID, "client_secret": CLIENT_SECRET},
        timeout=30,
    )
    r.raise_for_status()
    payload = r.json()
    _token["value"] = payload["access_token"]
    _token["expires_at"] = time.time() + float(payload.get("expires_in", 300))
    return _token["value"]


def tx_get(path, params):
    r = session.get(f"{NCTS_FHIR_BASE}/{path}", params=params,
                    headers={"Authorization": f"Bearer {get_token()}"}, timeout=60)
    r.raise_for_status()
    return r.json()


if ONLINE:
    get_token()
    print("Token obtenido; vence en", round(_token["expires_at"] - time.time()), "s")

### Paso 7 — Leer un recurso `Parameters` de FHIR

Las operaciones terminológicas responden con un recurso `Parameters`: una lista de pares nombre-valor, donde el valor viaja en un campo cuyo nombre empieza por `value` (`valueString`, `valueBoolean`, `valueCode`, `valueCoding`...) y los grupos complejos (`designation`, `property`) se descomponen en `part`.

**Explicación línea por línea**

- `def _value(p):` devuelve el contenido del primer campo que empieza por `value`, sin importar su tipo.
- `def params_to_dict(parameters):` convierte la lista en un diccionario práctico.
- `if name == "designation":` cada designación (por ejemplo el nombre completamente especificado, FSN) se guarda como diccionario de sus `part`.
- `elif name == "property":` las propiedades (`parent`, `inactive`...) se agrupan por `code` en listas, porque un concepto puede tener varios padres.
- `else: out[name] = _value(p)`: el resto (`display`, `name`, `version`, `result`, `message`) queda directo.
- El ejemplo final usa la estructura que publica la especificación FHIR R4 para `$lookup`; sirve para probar la función sin conexión.

In [ ]:
def _value(p):
    return next((v for k, v in p.items() if k.startswith("value")), None)


def params_to_dict(parameters):
    out = {"designation": [], "property": {}}
    for p in parameters.get("parameter", []):
        name = p.get("name")
        if name == "designation":
            out["designation"].append({q["name"]: _value(q) for q in p.get("part", [])})
        elif name == "property":
            parts = {q["name"]: _value(q) for q in p.get("part", [])}
            out["property"].setdefault(parts.get("code"), []).append(parts.get("value"))
        else:
            out[name] = _value(p)
    return out


ejemplo = {"resourceType": "Parameters", "parameter": [
    {"name": "name", "valueString": "LOINC"},
    {"name": "display", "valueString": "Bicarbonate [Moles/volume] in Serum"},
    {"name": "property", "part": [{"name": "code", "valueCode": "parent"}, {"name": "value", "valueCode": "LP15428-6"}]},
]}
params_to_dict(ejemplo)

### Paso 8 — `$lookup` de SNOMED CT con caché en disco y ritmo controlado

**Explicación línea por línea**

- `SNOMED = "http://snomed.info/sct"`: URI oficial del sistema SNOMED CT en FHIR. `SCT_AU = SNOMED + "/32506021000036107"`: identificador de la edición australiana (SNOMED CT-AU); se pasa como `version` para que la búsqueda use esa edición.
- `FSN_USE = "900000000000003001"`: código SNOMED del tipo de descripción "nombre completamente especificado" (FSN).
- `def lookup_snomed(code):` llama a `CodeSystem/$lookup` con `system`, `code` y `version`, y pide las propiedades `inactive` y `parent`.
- `except requests.HTTPError as exc: ... status_code in (400, 404)`: un código inexistente devuelve error; se registra como `found=False` en vez de detener el lote.
- `fsn = next((d["value"] for d in res["designation"] if (d.get("use") or {}).get("code") == FSN_USE), None)`: busca la designación cuyo uso es FSN.
- `CACHE = REF_DIR / "snomed_lookup_cache.parquet"`: los resultados se guardan; si vuelves a ejecutar, solo consultas lo que falta. **En la práctica real** las respuestas terminológicas se cachean localmente; la propia guía del NCTS no recomienda usar el servidor nacional como endpoint de producción.
- `time.sleep(0.2)`: pausa entre llamadas para no saturar un servicio público.
- `pendientes = [...][:MAX_LOOKUPS]`: limita el lote (50 códigos más frecuentes) para la primera ejecución; súbelo cuando todo funcione.

In [ ]:
SNOMED = "http://snomed.info/sct"
SCT_AU = SNOMED + "/32506021000036107"
FSN_USE = "900000000000003001"
MAX_LOOKUPS = 50


def lookup_snomed(code):
    try:
        res = params_to_dict(tx_get("CodeSystem/$lookup", {
            "system": SNOMED, "code": code, "version": SCT_AU, "property": ["inactive", "parent"]}))
    except requests.HTTPError as exc:
        if exc.response is not None and exc.response.status_code in (400, 404):
            return {"snomed_code": code, "found": False}
        raise
    fsn = next((d["value"] for d in res["designation"] if (d.get("use") or {}).get("code") == FSN_USE), None)
    inactive = res["property"].get("inactive", [False])[0]
    return {"snomed_code": code, "found": True, "display": res.get("display"), "fsn": fsn,
            "inactive": bool(inactive), "parents": ",".join(map(str, res["property"].get("parent", [])))}


CACHE = REF_DIR / "snomed_lookup_cache.parquet"
cache = pd.read_parquet(CACHE) if CACHE.exists() else pd.DataFrame(columns=["snomed_code"])

if ONLINE:
    pendientes = [c for c in snomed_inv["snomed_code"] if c not in set(cache["snomed_code"])][:MAX_LOOKUPS]
    nuevos = []
    for code in pendientes:
        nuevos.append(lookup_snomed(code))
        time.sleep(0.2)
    cache = pd.concat([cache, pd.DataFrame(nuevos)], ignore_index=True)
    cache.to_parquet(CACHE, index=False)
    print(f"Consultados ahora: {len(pendientes)} | en caché: {len(cache)}")
cache.head()

### Paso 9 — Comparar la descripción de Synthea con la terminología oficial

**Explicación línea por línea**

- `qa = snomed_inv.merge(cache, on="snomed_code", how="left")`: une inventario y resultados; `how="left"` conserva los códigos no consultados aún.
- `qa["fsn_matches"] = qa["description"].str.lower() == qa["fsn"].str.lower()`: Synthea a veces guarda el FSN (`Viral sinusitis (disorder)`) y otras una descripción corta (`Hypertension`). Comparar con el FSN oficial muestra cuáles difieren.
- `qa[qa["found"] == False]`: códigos que el servidor no reconoce.
- `qa[qa["inactive"] == True]`: conceptos **inactivos**: existieron pero fueron retirados de SNOMED CT; en un sistema real deben reemplazarse por su sucesor.
- `qa.to_parquet(REF_DIR / "terminology_qa.parquet")`: publica el resultado de la auditoría. Sin conexión, la tabla contiene solo las validaciones de la Parte A.

In [ ]:
qa = snomed_inv.merge(cache, on="snomed_code", how="left")
if "fsn" in qa.columns:
    qa["fsn_matches"] = qa["description"].str.lower() == qa["fsn"].str.lower()
    print("No encontrados:", int((qa["found"] == False).sum()), "| inactivos:", int((qa["inactive"] == True).sum()))
    print(qa.loc[qa["fsn_matches"] == False, ["snomed_code", "description", "fsn"]].head(10).to_string(index=False))
qa.to_parquet(REF_DIR / "terminology_qa.parquet", index=False)
print("Auditoría guardada:", len(qa), "códigos")

### Paso 10 — `$validate-code` para LOINC

El National Terminology Server también aloja LOINC.

**Explicación línea por línea**

- `def validate_loinc(code):` llama a `CodeSystem/$validate-code` con `url=http://loinc.org` y `code`.
- `res.get("result")`: booleano obligatorio de la respuesta; `display` y `message` son opcionales.
- `sospechosos = loinc_inv.loc[~loinc_inv["loinc_format_ok"], "loinc_code"].tolist()`: validas primero los que fallaron offline (deberían salir `False`) y luego 5 frecuentes como control (deberían salir `True`).
- `"5767-9"` se añade a propósito: en Synthea dos descripciones distintas (`Appearance of Urine` y `Odor of Urine`) usan ese mismo código. El `display` oficial te dirá cuál es correcto.

In [ ]:
LOINC = "http://loinc.org"


def validate_loinc(code):
    res = params_to_dict(tx_get("CodeSystem/$validate-code", {"url": LOINC, "code": code}))
    return {"loinc_code": code, "valid": bool(res.get("result")), "display": res.get("display"), "message": res.get("message")}


if ONLINE:
    sospechosos = loinc_inv.loc[~loinc_inv["loinc_format_ok"], "loinc_code"].tolist()
    controles = loinc_inv.loc[loinc_inv["loinc_format_ok"], "loinc_code"].head(5).tolist()
    filas = []
    for code in sospechosos + controles + ["5767-9"]:
        filas.append(validate_loinc(code))
        time.sleep(0.2)
    loinc_qa = pd.DataFrame(filas)
    print(loinc_qa.to_string(index=False))

## Parte C — Conjuntos de conceptos con ECL

### Paso 11 — `$expand` de un ValueSet implícito definido con ECL

En vez de buscar un único código ("hipertensión = 59621000"), la práctica estándar es definir un **conjunto de conceptos** con el *Expression Constraint Language* (ECL) de SNOMED CT: `<< 38341003` significa "38341003 y todos sus descendientes". FHIR R4 define ValueSets implícitos de SNOMED CT con la forma `http://snomed.info/sct?fhir_vs=ecl/<ECL>`, que Ontoserver expande con `ValueSet/$expand`.

**Explicación línea por línea**

- `CONCEPT_SETS = {...}`: nombre del conjunto → (expresión ECL, códigos semilla). `38341003` es *Hypertensive disorder, systemic arterial*; `44054006` es *Diabetes mellitus type 2*; `15777000` es *Prediabetes*. Los códigos semilla se usan como respaldo sin conexión.
- `def expand_ecl(ecl, page_size=500):` pide la expansión por páginas.
- `vs_url = f"{SCT_AU}?fhir_vs=ecl/{ecl}"`: ValueSet implícito sobre la edición australiana. `requests` se encarga de codificar `<<` y los espacios en la URL.
- `tx_get("ValueSet/$expand", {"url": vs_url, "count": page_size, "offset": offset})`: `count` y `offset` son los parámetros de paginación de `$expand`.
- `exp.get("contains", [])`: lista de conceptos de esta página; `exp.get("total")`: total de la expansión.
- `if not items or offset >= total: break`: termina al llegar al total.
- Sin conexión, cada conjunto contiene solo sus semillas y queda marcado `source = "seed_offline"`. **Importante:** un conjunto de semillas es una aproximación; documenta en tu README qué versión usaste.
- `concept_sets.to_parquet(...)`: la semana 7 leerá este archivo para definir el desenlace del modelo.

In [ ]:
CONCEPT_SETS = {
    "hypertension": ("<< 38341003", ["59621000"]),
    "t2dm_or_prediabetes": ("<< 44054006 OR << 15777000", ["44054006", "15777000"]),
}


def expand_ecl(ecl, page_size=500):
    vs_url = f"{SCT_AU}?fhir_vs=ecl/{ecl}"
    offset, out = 0, []
    while True:
        vs = tx_get("ValueSet/$expand", {"url": vs_url, "count": page_size, "offset": offset})
        exp = vs.get("expansion", {})
        items = exp.get("contains", [])
        out += [{"code": c["code"], "display": c.get("display")} for c in items]
        offset += len(items)
        total = exp.get("total", offset)
        if not items or offset >= total:
            break
    return out


filas = []
for nombre, (ecl, semillas) in CONCEPT_SETS.items():
    if ONLINE:
        miembros = expand_ecl(ecl)
        fuente = "ncts_expand"
    else:
        miembros = [{"code": c, "display": None} for c in semillas]
        fuente = "seed_offline"
    filas += [{"concept_set": nombre, "ecl": ecl, "code": m["code"], "display": m["display"], "source": fuente} for m in miembros]
    print(f"{nombre:<22} {len(miembros):>6} conceptos ({fuente})")

concept_sets = pd.DataFrame(filas)
concept_sets.to_parquet(REF_DIR / "concept_sets.parquet", index=False)

### Paso 12 — ¿Cuánto cambia una cohorte con un conjunto de conceptos?

**Explicación línea por línea**

- `for nombre, grupo in concept_sets.groupby("concept_set"):` recorre cada conjunto.
- `codigos = set(grupo["code"])`: miembros del conjunto.
- `n_set = cond.loc[cond["snomed_code"].isin(codigos), "patient_id"].nunique()`: pacientes con **cualquier** código del conjunto.
- `n_seed = ...isin(semillas)...`: pacientes con solo el código semilla.
- `codigos_en_datos = sorted(codigos & set(cond["snomed_code"]))`: qué miembros del conjunto aparecen de verdad en tus datos (por ejemplo, complicaciones como la retinopatía diabética son descendientes de diabetes tipo 2).
- **Por qué importa:** con datos reales, definir una enfermedad con un único código subestima la cohorte; la diferencia entre `n_seed` y `n_set` es exactamente ese sesgo.

In [ ]:
for nombre, grupo in concept_sets.groupby("concept_set"):
    codigos = set(grupo["code"])
    semillas = set(CONCEPT_SETS[nombre][1])
    n_set = cond.loc[cond["snomed_code"].isin(codigos), "patient_id"].nunique()
    n_seed = cond.loc[cond["snomed_code"].isin(semillas), "patient_id"].nunique()
    codigos_en_datos = sorted(codigos & set(cond["snomed_code"]))
    print(f"{nombre}: semillas={n_seed} pacientes | conjunto={n_set} pacientes | códigos presentes: {codigos_en_datos[:8]}")

### Autoevaluación

1. ¿Qué tipo de error detecta un dígito verificador y qué tipo de error **no** puede detectar?
2. ¿Por qué se cachean las respuestas del servidor terminológico?
3. Escribe en ECL: "infarto de miocardio y todos sus subtipos" (busca el código en el navegador Shrimp del NCTS).
4. ¿Qué riesgo tiene que `QALY` aparezca en una columna llamada `loinc_code`?

**Commit sugerido:** `feat(s05): auditoría terminológica SNOMED CT/LOINC y conjuntos de conceptos vía NCTS`

**Recuerda:** comprueba con `git status` que ningún archivo con credenciales entra al commit.